# Structured Output

requesting models to provide their response in a format matching a given schema, useful for ensuring the output can easily be parsed and used in subsequent processing

can be implemented using :
- Pydantic
- TypeDict
- Data Classes

# Pydantic

initializing the llm

In [1]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020C83047B60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020C831F4980>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str=Field(description="Title of the movie")
    year: int=Field(description="The year in which movie was released")
    director: str=Field(description="The director of the movie")
    ratings: float=Field(description="The movies rating out of 10")

Applying the output shcema to model

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020C83047B60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020C831F4980>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

comparing outputs from both

In [5]:
response = model.invoke("Provide details about the movie Dune")
response

AIMessage(content='<think>\nOkay, I need to provide details about the movie Dune. Let me start by recalling what I know. Dune is a science fiction film based on the novel by Frank Herbert. There are two main versions of the movie. The first one was released in 1984 directed by David Lynch. I think that one was not very well received. Then there\'s the more recent one from 2021 directed by Denis Villeneuve. I should mention both but focus more on the 2021 version since it\'s more recent and popular.\n\nFirst, I need to outline the basic information for both movies. For the 1984 version: director, release date, main cast like Kyle MacLachlan as Paul Atreides, Francesca Annis as Lady Jessica. The story follows Paul Atreides, a young man on the desert planet Arrakis. The movie is part of a two-part series, so the 1984 movie was split into two parts, but the second part was released as a TV movie called Dune: Part 2, which might not be well-regarded. The 1984 version is known for its ambiti

In [6]:
response = model_with_structure.invoke("Provide details about the movie Dune")
response

Movie(title='Dune', year=2021, director='Denis Villeneuve', ratings=8.5)

Nested Structure of PYdantic

In [7]:
from typing import List
class Actor(BaseModel):
    name: str = Field(description="Name of the actor/actress")
    role: str = Field(description="Role played by the actor in the movie")

class MovieDetails(BaseModel):
    title:str = Field(description="Title of the movie")
    year:int = Field(description="Year in which the movie was released")
    cast: List[Actor] = Field(description="List of actors in the movie")
    genres: List[str] = Field(description="List of genres in which the movie falls")
    budget: float | None = Field(None,description="Budget in millions USD")

model2 = model.with_structured_output(MovieDetails)
model2

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020C83047B60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020C831F4980>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDetails', 'description': '', 'parameters': {'properties': {'

In [8]:
response = model2.invoke("Provide details about the movie Dune")
response

MovieDetails(title='Dune', year=1984, cast=[Actor(name='Kyle MacLachlan', role='Paul Atreides'), Actor(name='Sean Young', role='Chani'), Actor(name='Jason Robards', role='Leto Atreides')], genres=['Science Fiction', 'Adventure'], budget=26.0)